# Keithley 2450 LAN check

Read-only: nothing here sources or enables the output. Run with **no junction on the probes**.

Set `IP` and run all cells. The last cell prints the `RESOURCE` string for the ABAA notebook.

In [7]:
import socket
import pyvisa

IP = "192.168.50.2"   # instrument address
BACKEND = ""          # "" = installed VISA (NI-VISA); "@py" = pyvisa-py

CANDIDATES = [
    f"TCPIP0::{IP}::inst0::INSTR",   # VXI-11
    f"TCPIP0::{IP}::5025::SOCKET",   # raw socket
]

In [8]:
# Network layer, independent of VISA
with socket.create_connection((IP, 5025), timeout=3):
    print(f"TCP OK: {IP}:5025")

TCP OK: 192.168.50.2:5025


In [9]:
rm = pyvisa.ResourceManager(BACKEND)
print(rm.visalib)

inst, RESOURCE = None, None
for r in CANDIDATES:
    try:
        s = rm.open_resource(r, open_timeout=5000)
        s.timeout = 5000
        s.write_termination = "\n"
        s.read_termination = "\n"
        print(f"{r}\n  {s.query('*IDN?').strip()}")
        inst, RESOURCE = s, r
        break
    except Exception as e:
        print(f"{r}\n  failed: {e}")

assert inst is not None, "No VISA connection"

Visa Library at C:\WINDOWS\system32\visa32.dll
TCPIP0::192.168.50.2::inst0::INSTR
  KEITHLEY INSTRUMENTS,MODEL 2450,MYFP002618,1.7.16a


In [10]:
lang = inst.query("*LANG?").strip()
print("Command set:", lang)
assert lang == "SCPI", "Set Command Set to SCPI on the front panel; the instrument restarts"

print("Terminals:  ", inst.query(":ROUT:TERM?").strip())
print("Output:     ", "ON" if int(inst.query(":OUTP?")) else "off")
print("Error queue:", inst.query(":SYST:ERR?").strip())

Command set: SCPI
Terminals:   FRON
Output:      off
Error queue: 0,"No error;0;0 0"


In [11]:
inst.close()
rm.close()
print(f'RESOURCE = "{RESOURCE}"')

RESOURCE = "TCPIP0::192.168.50.2::inst0::INSTR"
